In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import seaborn as sns

In [ ]:
df = pd.read_csv("train.csv")

In [ ]:
df.loc[:,"Embarked"].value_counts()

In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit

split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

for train_indices, test_indices in split.split(
        df, 
        df[["Survived", "Pclass", "Sex"]]):
    
    strat_train_set = df.loc[train_indices]
    strat_test_set = df.loc[test_indices]

In [ ]:
strat_train_set.isna().sum()

In [ ]:
plt.subplot(1,2,1)
strat_train_set["Survived"].hist()
strat_train_set["Pclass"].hist()

plt.subplot(1,2,2)
strat_test_set["Survived"].hist()
strat_test_set["Pclass"].hist()

In [ ]:
sns.heatmap( df.corr(numeric_only=True), cmap="coolwarm")
plt.show()

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.impute import SimpleImputer
class AgeImputer(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.imputer = SimpleImputer(strategy = "mean")
    def fit(self, X, y=None):
        self.imputer.fit(X[["Age"]])
        return self
    def transform(self, X):
        X = X.copy()
        X["Age"] = self.imputer.transform(X[["Age"]]).ravel()
        return X

In [ ]:
from sklearn.preprocessing import OneHotEncoder
class FeatureEncoder(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.encoder = OneHotEncoder(handle_unknown= "ignore")
    def fit(self, X, y=None):
        self.encoder.fit(X[["Embarked", "Sex"]])
        return self
    def transform(self, X):
        X = X.copy()
        encoded = self.encoder.transform(X[["Embarked", "Sex"]]).toarray()
        col_names = self.encoder.get_feature_names_out(["Embarked", "Sex"])
        for i, name in enumerate(col_names):
            X[name] = encoded[:, i]
        return X

In [ ]:
class FeaturDropper(BaseEstimator, TransformerMixin):
    def fit(self, X, y = None):
        return self
    def transform(self, X):
        return X.drop(["Embarked_nan","Embarked","Name", "Ticket", "Cabin", "Sex"], axis = 1, errors = "ignore")

In [ ]:
from sklearn.pipeline import Pipeline
pipeline = Pipeline([
    ("ageimputer", AgeImputer()),
    ("featureencoder", FeatureEncoder()),
    ("featuredropper", FeaturDropper())
])

In [ ]:
df.isna().sum()

In [ ]:
final_data = pipeline.fit_transform(df)

In [ ]:
final_data

In [ ]:
from sklearn.preprocessing import StandardScaler
X_final = final_data.drop(["Survived"], axis = 1)
y_final = final_data["Survived"]

scaler = StandardScaler()
X_data_final = scaler.fit_transform(X_final) 
y_data_final = y_final.to_numpy()

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
prod_clf = RandomForestClassifier()
param_grid = [
    {"n_estimators": [10, 100, 200, 500], "max_depth": [None, 5, 10], "min_samples_split": [2,3,4]}
]
grid_search = GridSearchCV(prod_clf, param_grid, cv=3, scoring="accuracy", return_train_score=True)
grid_search.fit(X_data_final, y_data_final)

In [ ]:
grid_search.best_score_

In [ ]:
prod_final_clf = grid_search.best_estimator_

In [ ]:
prod_final_clf.score(X_data_final, y_data_final)

In [ ]:
df_test = pd.read_csv("test.csv")

In [ ]:
test_data = pipeline.transform(df_test)

In [ ]:
X_test_data = scaler.transform(test_data) 

In [ ]:
X_test_data.shape

In [ ]:
prediction = prod_final_clf.predict(X_test_data)

In [ ]:
submission =  df_test[["PassengerId"]].copy()
submission["Survived"] = prediction


In [ ]:
submission

In [ ]:
submission.to_csv("submission.csv", index=False)